In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from AlignmentTechniques import LatentAlignment2d, AdaptiveBatchNorm2d, EuclideanAlignment

class SeizureConvNet(nn.Module):
    """
    Deep Convolutional Neural Network inspired by Deep4Net architectures,
    optimized for high-frequency anomaly extraction (Seizure Detection).
    """
    def __init__(self, in_shape, n_out, alignment='None', dropout=0.5):
        super(SeizureConvNet, self).__init__()
        self.in_shape = in_shape
        self.n_out = n_out
        self.alignment = alignment

        # Base number of filters that doubles at each layer
        n_filters = 25

        if alignment == 'euclidean':
            self.euclidean = EuclideanAlignment()

        # Input BatchNorm / Latent Alignment
        if alignment == 'latent':
            self.latent_align0 = LatentAlignment2d(in_shape[0], affine=False)
        elif alignment == 'adaptive':
            self.abn0 = AdaptiveBatchNorm2d(in_shape[0], affine=False)
        else:
            self.bn0 = nn.BatchNorm2d(in_shape[0], affine=False)

        # --- BLOCK 1: Temporal-Spatial Extraction ---
        self.conv1_time = nn.Conv2d(1, n_filters, kernel_size=(1, 11), padding=(0, 5), bias=False)
        self.conv1_space = nn.Conv2d(n_filters, n_filters, kernel_size=(in_shape[0], 1), bias=False)
        if alignment == 'latent':
            self.latent_align1 = LatentAlignment2d(n_filters)
        elif alignment == 'adaptive':
            self.abn1 = AdaptiveBatchNorm2d(n_filters)
        else:
            self.bn1 = nn.BatchNorm2d(n_filters)
        self.pool1 = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))
        self.drop1 = nn.Dropout(dropout)

        # --- BLOCK 2: Deep Feature Extraction ---
        self.conv2 = nn.Conv2d(n_filters, n_filters * 2, kernel_size=(1, 11), padding=(0, 5), bias=False)
        if alignment == 'latent':
            self.latent_align2 = LatentAlignment2d(n_filters * 2)
        elif alignment == 'adaptive':
            self.abn2 = AdaptiveBatchNorm2d(n_filters * 2)
        else:
            self.bn2 = nn.BatchNorm2d(n_filters * 2)
        self.pool2 = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))
        self.drop2 = nn.Dropout(dropout)

        # --- BLOCK 3: High-level Features ---
        self.conv3 = nn.Conv2d(n_filters * 2, n_filters * 4, kernel_size=(1, 11), padding=(0, 5), bias=False)
        if alignment == 'latent':
            self.latent_align3 = LatentAlignment2d(n_filters * 4)
        elif alignment == 'adaptive':
            self.abn3 = AdaptiveBatchNorm2d(n_filters * 4)
        else:
            self.bn3 = nn.BatchNorm2d(n_filters * 4)
        self.pool3 = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))
        self.drop3 = nn.Dropout(dropout)

        # --- CLASSIFIER ---
        # Dynamic calculation of the residual time dimension after 3 Max Pooling layers
        final_time = in_shape[1] // 3 // 3 // 3
        self.n_features = n_filters * 4 * final_time
        self.fc_out = nn.Linear(self.n_features, n_out)

    def forward(self, x, sbj_trials):
        """
        Args:
             x: (batch * sbj_trials, spatial, time)
             sbj_trials: number of trials per subject
        """
        _, spatial, time = x.shape

        if self.alignment == 'euclidean':
            x = self.euclidean(x, sbj_trials)

        x = x.reshape(-1, spatial, 1, time)
        if self.alignment == 'latent':
            x = self.latent_align0(x, sbj_trials)
        elif self.alignment == 'adaptive':
            x = self.abn0(x, sbj_trials)
        else:
            x = self.bn0(x)
        x = x.reshape(-1, spatial, time)

        x = x.unsqueeze(1) # Artificial image channel (batch, 1, spatial, time)

        # Block 1
        x = self.conv1_time(x)
        x = self.conv1_space(x)
        if self.alignment == 'latent':
            x = self.latent_align1(x, sbj_trials)
        elif self.alignment == 'adaptive':
            x = self.abn1(x, sbj_trials)
        else:
            x = self.bn1(x)
        x = F.elu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        # Block 2
        x = self.conv2(x)
        if self.alignment == 'latent':
            x = self.latent_align2(x, sbj_trials)
        elif self.alignment == 'adaptive':
            x = self.abn2(x, sbj_trials)
        else:
            x = self.bn2(x)
        x = F.elu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        # Block 3
        x = self.conv3(x)
        if self.alignment == 'latent':
            x = self.latent_align3(x, sbj_trials)
        elif self.alignment == 'adaptive':
            x = self.abn3(x, sbj_trials)
        else:
            x = self.bn3(x)
        x = F.elu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        x = x.reshape(-1, self.n_features)
        x = self.fc_out(x)

        return x

In [ ]:
import os
import urllib.request
import mne
import numpy as np
import pandas as pd
import gc

# 1. Manual mapping of files and seizure timings (start_s, end_s)
# This avoids having to download and parse complex "summary" text files
epilepsy_data_map = {
    1: {'file': 'chb01_03.edf', 'url': 'https://physionet.org/files/chbmit/1.0.0/chb01/chb01_03.edf', 'seizures': [(2996, 3036)]},
    2: {'file': 'chb02_16.edf', 'url': 'https://physionet.org/files/chbmit/1.0.0/chb02/chb02_16.edf', 'seizures': [(130, 212)]},
    3: {'file': 'chb03_01.edf', 'url': 'https://physionet.org/files/chbmit/1.0.0/chb03/chb03_01.edf', 'seizures': [(362, 414)]},
    4: {'file': 'chb05_06.edf', 'url': 'https://physionet.org/files/chbmit/1.0.0/chb05/chb05_06.edf', 'seizures': [(417, 532)]}
}

common_channels = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1', 'FP1-F3', 'F3-C3', 
    'C3-P3', 'P3-O1', 'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2', 'FZ-CZ', 'CZ-PZ'
]

def extract_epilepsy_data(subject_id, info):
    filename = info['file']
    # Download the file only if it is not already present in the folder
    if not os.path.exists(filename):
        print(f"    Downloading {filename} from PhysioNet (approx. 30-50MB)...")
        urllib.request.urlretrieve(info['url'], filename)

    # Load raw file
    raw = mne.io.read_raw_edf(filename, preload=True, verbose=False)
    
    # --- NEW CHANNEL MANAGEMENT LOGIC ---
    current_picks = []
    rename_dict = {}

    for target_ch in common_channels:
            # Search among file names for the one matching the target (or having suffixes)
            found = [ch for ch in raw.ch_names if ch == target_ch or ch.startswith(target_ch + '-')]
            if found:
                # Take only the first match (e.g., FP1-F7-0 becomes FP1-F7)
                actual_name = found[0]
                current_picks.append(actual_name)
                rename_dict[actual_name] = target_ch

    raw.pick(current_picks)
    raw.rename_channels(rename_dict)
    
    # For safety, if channels are missing compared to standard, we reorder or handle consistency
    raw.reorder_channels([ch for ch in common_channels if ch in raw.ch_names])
    
    # Resample to 100 Hz to lighten the neural network and standardize data
    raw.resample(100.0, verbose=False)

    # Create 4-second epochs (400 samples at 100 Hz)
    epochs = mne.make_fixed_length_epochs(raw, duration=4.0, preload=True, verbose=False)
    X = epochs.get_data() * 1e6  # Conversion to microvolts
    
    # Initialize labels to 0 (Background / No seizure)
    y = np.zeros(len(X), dtype=np.int64)
    
    # Start times for each epoch in seconds
    epoch_times = epochs.events[:, 0] / raw.info['sfreq'] 
    
    # Labeling: if the epoch overlaps with the seizure period, label as 1 (Target)
    for (start_sz, end_sz) in info['seizures']:
        for i, t in enumerate(epoch_times):
            if (t + 4.0) > start_sz and t < end_sz:
                y[i] = 1
                
    return X, y

# --- EXTRACTION EXECUTION ---
X_list, y_list, meta_list = [], [], []

print("Starting Epilepsy Data Extraction (CHB-MIT)...")
for sbj, info in epilepsy_data_map.items():
    print(f"Processing Patient {sbj}...")
    X_tmp, y_tmp = extract_epilepsy_data(sbj, info)
    
    X_list.append(X_tmp.astype('float32'))
    y_list.append(y_tmp)
    meta_list.append(pd.DataFrame({'subject': [sbj] * len(y_tmp)}))
    
    del X_tmp
    gc.collect()

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)
metadata = pd.concat(meta_list, ignore_index=True)

print(f"\n Loading and Preparation Completed")
print(f"Final shape (X): {X.shape} -> (trials, 18 channels, 400 samples)")

# Class imbalance analysis
n_seizures = np.sum(y == 1)
n_background = np.sum(y == 0)
print(f"\n--- IMBALANCE ANALYSIS ---")
print(f"Background Epochs (Class 0): {n_background}")
print(f"Seizure Epochs (Class 1): {n_seizures}")
print(f"Ratio: approximately {n_background // n_seizures} to 1")

Inizio Estrazione Dati Epilessia (CHB-MIT)...
Elaborazione Paziente 1...
   Scaricando chb01_03.edf da PhysioNet (circa 30-50MB)...


/tmp/ipykernel_18963/2751087627.py:25: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(filename, preload=True, verbose=False)


Elaborazione Paziente 2...
   Scaricando chb02_16.edf da PhysioNet (circa 30-50MB)...


/tmp/ipykernel_18963/2751087627.py:25: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(filename, preload=True, verbose=False)


Elaborazione Paziente 3...
   Scaricando chb03_01.edf da PhysioNet (circa 30-50MB)...


/tmp/ipykernel_18963/2751087627.py:25: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(filename, preload=True, verbose=False)


Elaborazione Paziente 4...
   Scaricando chb05_06.edf da PhysioNet (circa 30-50MB)...


/tmp/ipykernel_18963/2751087627.py:25: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(filename, preload=True, verbose=False)



✅ Caricamento e Preparazione Completati!
Shape finale (X): (2939, 18, 400) -> (prove, 18 canali, 400 campioni)

--- ANALISI SBILANCIAMENTO ---
Epoche Background (Classe 0): 2865
Epoche Crisi (Classe 1): 74
Rapporto: circa 38 a 1!


In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import balanced_accuracy_score, f1_score
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).long()

n_channels = X.shape[1]
n_samples = X.shape[2]
n_classes = len(np.unique(y))

# --- LOSS WEIGHT CALCULATION FOR IMBALANCED DATA ---
class_counts = np.bincount(y)
weights = 1.0 / torch.tensor(class_counts, dtype=torch.float32)
weights = weights / weights.sum() * 2.0 # Normalization
weights = weights.to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

print(f"Loss weights applied: Background={weights[0]:.4f}, Seizure={weights[1]:.4f}")

# --- LOSO TRAINING LOOP ---
print("\n=== Starting LOSO Validation on Epilepsy (SeizureConvNet + Latent Alignment) ===")
loso_bal_acc = []
loso_f1 = []
subjects_array = metadata['subject'].unique()
num_epochs = 30 # Deep CNNs require a few more epochs

for test_subject in subjects_array:
    print(f"\n-> Training for Test Subject: {test_subject}")
    
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values
    
    model = SeizureConvNet(in_shape=(n_channels, n_samples), n_out=n_classes, alignment='latent').to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_subjects = metadata[train_mask]['subject'].unique()
    
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for subj in train_subjects:
            subj_mask = (metadata['subject'] == subj).values
            batch_X = X_tensor[subj_mask].to(device)
            batch_y = y_tensor[subj_mask].to(device)
            
            optimizer.zero_grad(set_to_none=True)
            
            # Deep Sets approach: passing the entire subject
            outputs = model(batch_X, sbj_trials=batch_X.shape[0])
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()

            del batch_X, batch_y
            
        if (epoch + 1) % 10 == 0:
            print(f"   Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_subjects):.4f}")
            
    # --- EVALUATION ---
    model.eval()
    with torch.no_grad():
        test_X = X_tensor[test_mask].to(device)
        test_y = y_tensor[test_mask]
        
        outputs = model(test_X, sbj_trials=test_X.shape[0])
        _, predicted = torch.max(outputs.data, 1)
        
        y_true = test_y.cpu().numpy()
        y_pred = predicted.cpu().numpy()
        
        # Robust metrics calculation
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        
        loso_bal_acc.append(bal_acc)
        loso_f1.append(f1)
        
    print(f"-> Finished Test Subject {test_subject} | Balanced Acc: {bal_acc*100:.2f}% | F1-Score: {f1:.4f} | Time: {time.time()-start_time:.1f}s")

print("\n" + "="*60)
print(f"FINAL LOSO RESULTS - Epilepsy (Latent Alignment)")
print(f"Mean Balanced Accuracy: {np.mean(loso_bal_acc)*100:.2f}% ± {np.std(loso_bal_acc)*100:.2f}%")
print(f"Mean F1-Score:          {np.mean(loso_f1):.4f} ± {np.std(loso_f1):.4f}")
print("="*60)


```text
Utilizzando il device: cuda
Pesi applicati alla Loss: Background=0.0504, Crisi=1.9496

=== Inizio Validazione LOSO su Epilessia (SeizureConvNet + Latent Alignment) ===

-> Addestramento per Test Subject: 1
   Epoca [10/30], Loss: 0.1880
   Epoca [20/30], Loss: 0.0532
   Epoca [30/30], Loss: 0.0195
-> Fine Test Subject 1 | Balanced Acc: 97.58% | F1-Score: 0.3175 | Tempo: 19.8s

-> Addestramento per Test Subject: 2
   Epoca [10/30], Loss: 0.0705
   Epoca [20/30], Loss: 0.0388
   Epoca [30/30], Loss: 0.0165
-> Fine Test Subject 2 | Balanced Acc: 88.73% | F1-Score: 0.6316 | Tempo: 22.7s

-> Addestramento per Test Subject: 3
   Epoca [10/30], Loss: 0.0954
   Epoca [20/30], Loss: 0.0446
   Epoca [30/30], Loss: 0.0160
-> Fine Test Subject 3 | Balanced Acc: 92.82% | F1-Score: 0.2857 | Tempo: 17.9s

-> Addestramento per Test Subject: 4
   Epoca [10/30], Loss: 0.1755
   Epoca [20/30], Loss: 0.0551
   Epoca [30/30], Loss: 0.0356
-> Fine Test Subject 4 | Balanced Acc: 97.07% | F1-Score: 0.5321 | Tempo: 18.0s

============================================================
RISULTATI FINALI LOSO Epilessia (Latent Alignment)
Balanced Accuracy Media: 94.05% ± 3.59%
F1-Score Medio:          0.4417 ± 0.1449
============================================================


In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import balanced_accuracy_score, f1_score
import time

print("\n=== Starting LOSO Validation on Epilepsy (BASELINE WITHOUT ALIGNMENT) ===")
loso_bal_acc_base = []
loso_f1_base = []

for test_subject in subjects_array:
    print(f"\n-> Training for Test Subject: {test_subject}")
    
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values
    
    # Initialize the model with alignment='None'
    model = SeizureConvNet(in_shape=(n_channels, n_samples), n_out=n_classes, alignment='None').to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_subjects = metadata[train_mask]['subject'].unique()
    
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        # We maintain training by subject blocks for batching fairness
        for subj in train_subjects:
            subj_mask = (metadata['subject'] == subj).values
            batch_X = X_tensor[subj_mask].to(device)
            batch_y = y_tensor[subj_mask].to(device)
            
            optimizer.zero_grad(set_to_none=True)
            
            outputs = model(batch_X, sbj_trials=batch_X.shape[0])
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()

            del batch_X, batch_y
            
        if (epoch + 1) % 10 == 0:
            print(f"   Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_subjects):.4f}")
            
    # --- BASELINE EVALUATION ---
    model.eval()
    with torch.no_grad():
        test_X = X_tensor[test_mask].to(device)
        test_y = y_tensor[test_mask]
        
        outputs = model(test_X, sbj_trials=test_X.shape[0])
        _, predicted = torch.max(outputs.data, 1)
        
        y_true = test_y.cpu().numpy()
        y_pred = predicted.cpu().numpy()
        
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        
        loso_bal_acc_base.append(bal_acc)
        loso_f1_base.append(f1)
        
    print(f"-> Finished Test Subject {test_subject} | Balanced Acc: {bal_acc*100:.2f}% | F1-Score: {f1:.4f} | Time: {time.time()-start_time:.1f}s")

print("\n" + "="*60)
print(f"FINAL LOSO RESULTS - Epilepsy (BASELINE)")
print(f"Mean Balanced Accuracy: {np.mean(loso_bal_acc_base)*100:.2f}% ± {np.std(loso_bal_acc_base)*100:.2f}%")
print(f"Mean F1-Score:          {np.mean(loso_f1_base):.4f} ± {np.std(loso_f1_base):.4f}")
print("="*60)

```text
=== Inizio Validazione LOSO su Epilessia (BASELINE SENZA ALLINEAMENTO) ===

-> Addestramento per Test Subject: 1
   Epoca [10/30], Loss: 0.1127
   Epoca [20/30], Loss: 0.0671
   Epoca [30/30], Loss: 0.0297
-> Fine Test Subject 1 | Balanced Acc: 70.00% | F1-Score: 0.5714 | Tempo: 19.3s

-> Addestramento per Test Subject: 2
   Epoca [10/30], Loss: 0.0727
   Epoca [20/30], Loss: 0.0392
   Epoca [30/30], Loss: 0.0361
-> Fine Test Subject 2 | Balanced Acc: 83.08% | F1-Score: 0.4086 | Tempo: 21.8s

-> Addestramento per Test Subject: 3
   Epoca [10/30], Loss: 0.1063
   Epoca [20/30], Loss: 0.0325
   Epoca [30/30], Loss: 0.0129
-> Fine Test Subject 3 | Balanced Acc: 53.51% | F1-Score: 0.1250 | Tempo: 17.3s

-> Addestramento per Test Subject: 4
   Epoca [10/30], Loss: 0.1503
   Epoca [20/30], Loss: 0.0898
   Epoca [30/30], Loss: 0.0250
-> Fine Test Subject 4 | Balanced Acc: 96.44% | F1-Score: 0.6292 | Tempo: 17.9s

============================================================
RISULTATI FINALI LOSO Epilessia (BASELINE)
Balanced Accuracy Media: 75.76% ± 15.88%
F1-Score Medio:          0.4336 ± 0.1957
============================================================